# Model Training

In [31]:
from pyspark.ml.feature import VectorAssembler, StringIndexer
from pyspark.ml.classification import LogisticRegression, DecisionTreeClassifier, RandomForestClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator, BinaryClassificationEvaluator

In [34]:
spark = (
    SparkSession.builder
    .appName("BusServiceReliability")
    .master("local[*]")
    .getOrCreate()
)

In [35]:
df = spark.read.parquet("../outputs/cleaned_timetable_parquet")

In [36]:
df.show(5, truncate=False)

+-------------------------------------------------------------------------------------------------------+----------+----------+------------+-------------+----------+
|filename                                                                                               |journey_id|operator  |service_code|stop_sequence|route_size|
+-------------------------------------------------------------------------------------------------------+----------+----------+------------+-------------+----------+
|10A-None--SCMY-GM-2026-07-26-Gillmoss_July_2026_EV_ADDED_FINAL__SCMY_PC1033334_5_20260719-BODS_V1_1.xml|VJ2904    |Stagecoach|PC1033334:5 |55           |Long      |
|10A-None--SCMY-GM-2026-07-26-Gillmoss_July_2026_EV_ADDED_FINAL__SCMY_PC1033334_5_20260719-BODS_V1_1.xml|VJ2904    |Stagecoach|PC1033334:5 |56           |Long      |
|10A-None--SCMY-GM-2026-07-26-Gillmoss_July_2026_EV_ADDED_FINAL__SCMY_PC1033334_5_20260719-BODS_V1_1.xml|VJ2904    |Stagecoach|PC1033334:5 |57           |Long      |
|10A

## Preparing the Target Variable

In [37]:
from pyspark.sql.functions import when, col

df = df.withColumn(
    "target",
    when(col("route_size") == "Long", 1)
    .otherwise(0)
)

df.groupBy("target").count().show()

+------+------+
|target| count|
+------+------+
|     1| 37157|
|     0|234395|
+------+------+



In [38]:
df.select(
    "route_size",
    "target"
).show(10)

+----------+------+
|route_size|target|
+----------+------+
|      Long|     1|
|      Long|     1|
|      Long|     1|
|      Long|     1|
|      Long|     1|
|      Long|     1|
|      Long|     1|
|      Long|     1|
|      Long|     1|
|      Long|     1|
+----------+------+
only showing top 10 rows


## Feature Preparation

In [22]:
from pyspark.ml.feature import StringIndexer, VectorAssembler

In [39]:
indexer = StringIndexer(
    inputCol="service_code",
    outputCol="service_code_index"
)

df = indexer.fit(df).transform(df)

In [40]:
assembler = VectorAssembler(
    inputCols=[
        "stop_sequence",
        "service_code_index"
    ],
    outputCol="features"
)

dataset = assembler.transform(df)

dataset.select(
    "features",
    "target"
).show(5, truncate=False)

+----------+------+
|features  |target|
+----------+------+
|[55.0,3.0]|1     |
|[56.0,3.0]|1     |
|[57.0,3.0]|1     |
|[58.0,3.0]|1     |
|[59.0,3.0]|1     |
+----------+------+
only showing top 5 rows


## Train-Test Split

In [41]:
train_data, test_data = dataset.randomSplit([0.8, 0.2], seed=42)

print("Training records:", train_data.count())
print("Testing records:", test_data.count())

Training records: 217633
Testing records: 53919


In [45]:
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator, BinaryClassificationEvaluator
import pandas as pd

In [47]:
# Train Random Forest (Increased maxBins to handle 122 categorical values)
rf = RandomForestClassifier(featuresCol="features", labelCol="target", numTrees=20, seed=42, maxBins=128)
rf_model = rf.fit(train_data)

# Make predictions on test data
rf_predictions = rf_model.transform(test_data)

print("Random Forest trained successfully!")
rf_predictions.select("features", "target", "prediction", "probability").show(10, truncate=False)

Random Forest trained successfully!
+----------+------+----------+-----------+
|features  |target|prediction|probability|
+----------+------+----------+-----------+
|[57.0,3.0]|1     |1.0       |[0.0,1.0]  |
|[61.0,3.0]|1     |1.0       |[0.0,1.0]  |
|[63.0,3.0]|1     |1.0       |[0.0,1.0]  |
|[68.0,3.0]|1     |1.0       |[0.0,1.0]  |
|[6.0,3.0] |0     |0.0       |[1.0,0.0]  |
|[10.0,3.0]|0     |0.0       |[1.0,0.0]  |
|[16.0,3.0]|0     |0.0       |[1.0,0.0]  |
|[22.0,3.0]|0     |0.0       |[1.0,0.0]  |
|[32.0,3.0]|0     |0.0       |[1.0,0.0]  |
|[33.0,3.0]|0     |0.0       |[1.0,0.0]  |
+----------+------+----------+-----------+
only showing top 10 rows


In [49]:
# Re-train Logistic Regression just to be safe
print("Re-training Logistic Regression...")
lr = LogisticRegression(featuresCol="features", labelCol="target", maxIter=10)
lr_model = lr.fit(train_data)
lr_predictions = lr_model.transform(test_data)

# Re-train Random Forest just to be safe (with maxBins=128 to fix the error)
print("Re-training Random Forest...")
rf = RandomForestClassifier(featuresCol="features", labelCol="target", numTrees=20, seed=42, maxBins=128)
rf_model = rf.fit(train_data)
rf_predictions = rf_model.transform(test_data)

# Now save both models
print("Saving models...")
lr_model.save("logistic_regression_model")
print("Logistic Regression model saved successfully!")

rf_model.save("random_forest_model")
print("Random Forest model saved successfully!")

Re-training Logistic Regression...


26/08/02 00:36:41 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS
                                                                                

Re-training Random Forest...


Saving models...
Logistic Regression model saved successfully!
Random Forest model saved successfully!


## Train Decision Tree Classifier

In [26]:
from pyspark.ml.classification import DecisionTreeClassifier

dt = DecisionTreeClassifier(
    featuresCol="features",
    labelCol="target"
)

model = dt.fit(train_data)

## Make Predictions

In [42]:
predictions = model.transform(test_data)

In [43]:
predictions.select(
    "stop_sequence",
    "service_code",
    "target",
    "prediction",
    "probability"
).show(10, truncate=False)

+-------------+------------+------+----------+-----------+
|stop_sequence|service_code|target|prediction|probability|
+-------------+------------+------+----------+-----------+
|57           |PC1033334:5 |1     |0.0       |[1.0,0.0]  |
|61           |PC1033334:5 |1     |0.0       |[1.0,0.0]  |
|63           |PC1033334:5 |1     |0.0       |[1.0,0.0]  |
|68           |PC1033334:5 |1     |0.0       |[1.0,0.0]  |
|6            |PC1033334:5 |0     |0.0       |[1.0,0.0]  |
|10           |PC1033334:5 |0     |0.0       |[1.0,0.0]  |
|16           |PC1033334:5 |0     |0.0       |[1.0,0.0]  |
|22           |PC1033334:5 |0     |0.0       |[1.0,0.0]  |
|32           |PC1033334:5 |0     |0.0       |[1.0,0.0]  |
|33           |PC1033334:5 |0     |0.0       |[1.0,0.0]  |
+-------------+------------+------+----------+-----------+
only showing top 10 rows


## Feature Importance

In [44]:
print("Feature Importances:")
print(model.featureImportances)

Feature Importances:
(3,[0,1],[0.9409660107334527,0.05903398926654745])


## Save Trained Model

In [30]:
model.write().overwrite().save("../models/decision_tree_model")

print("Decision Tree model saved successfully!")

Decision Tree model saved successfully!


# Summary

In this notebook, the cleaned dataset was prepared for machine learning by creating a target variable and assembling numerical features. A Decision Tree classifier was trained using the training dataset, predictions were generated for the testing dataset, feature importance was examined, and the trained model was saved for evaluation.